## Bước 6: Feature Engineering
### 6.1 Đọc dữ liệu

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/depression_severity_preprocessed.csv')

print("Số dòng:", len(df))
print(df[['text_classical', 'label']].head(3))

Số dòng: 3519
                                      text_classical    label
0  said not felt way suggeted go rest trigger ahe...     mild
1  hey assistance not sure right place post go cu...  minimum
2  mom hit newspaper shocked would know don't lik...  minimum


### 6.2 TF-IDF Vectorization (cho LogReg, SVM)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

X_tfidf = tfidf_vectorizer.fit_transform(df['text_classical'])

print("Kích thước ma trận TF-IDF:", X_tfidf.shape)
print("Số chiều đặc trưng:", len(tfidf_vectorizer.get_feature_names_out()))
print("\n10 đặc trưng đầu tiên:", tfidf_vectorizer.get_feature_names_out()[:10])

Kích thước ma trận TF-IDF: (3519, 3000)
Số chiều đặc trưng: 3000

10 đặc trưng đầu tiên: ['ability' 'able' 'able afford' 'able get' 'abroad' 'absolute'
 'absolutely' 'abuse' 'abused' 'abuser']


### 6.3 Đặc trưng thủ công (cho XGBoost)

In [3]:
import re

def extract_handcrafted_features(text):
    text_lower = text.lower()
    words = text_lower.split()
    n_words = len(words) if len(words) > 0 else 1

    features = {}

    # 1. Độ dài bài đăng
    features['word_count'] = len(words)

    # 2. Tỉ lệ đại từ ngôi thứ nhất số ít (dấu hiệu tập trung vào bản thân)
    first_person = ['i', 'me', 'my', 'mine', 'myself']
    features['first_person_ratio'] = sum(words.count(w) for w in first_person) / n_words

    # 3. Tỉ lệ từ phủ định
    negation = ['not', 'no', 'never', "n't", 'nothing', 'none']
    features['negation_ratio'] = sum(words.count(w) for w in negation) / n_words

    # 4. Tỉ lệ từ tuyệt đối hóa (absolutist words — liên quan trầm cảm trong nghiên cứu tâm lý)
    absolutist = ['always', 'never', 'everyone', 'nobody', 'everything', 'nothing',
                   'completely', 'totally', 'entirely']
    features['absolutist_ratio'] = sum(words.count(w) for w in absolutist) / n_words

    # 5. Số câu (dựa trên dấu chấm câu)
    features['sentence_count'] = len(re.findall(r'[.!?]+', text))

    # 6. Tỉ lệ dấu chấm than (cảm xúc mạnh)
    features['exclamation_ratio'] = text.count('!') / max(len(text), 1)

    # 7. Từ khóa cảnh báo trực tiếp (dựa trên EDA đã làm ở Bước 4)
    warning_words = ['suicide', 'suicidal', 'kill', 'die', 'death', 'worthless', 'hopeless']
    features['warning_word_count'] = sum(text_lower.count(w) for w in warning_words)

    return features

# Áp dụng cho toàn bộ dataset (dùng text gốc, chưa lemmatize, để giữ đúng ngữ cảnh)
handcrafted = df['text'].apply(extract_handcrafted_features).apply(pd.Series)

print("Các đặc trưng thủ công:", list(handcrafted.columns))
print(handcrafted.head())

Các đặc trưng thủ công: ['word_count', 'first_person_ratio', 'negation_ratio', 'absolutist_ratio', 'sentence_count', 'exclamation_ratio', 'warning_word_count']
   word_count  first_person_ratio  negation_ratio  absolutist_ratio  \
0       113.0            0.079646        0.008850          0.000000   
1       108.0            0.009259        0.009259          0.009259   
2       166.0            0.090361        0.006024          0.000000   
3       273.0            0.146520        0.007326          0.007326   
4        89.0            0.078652        0.011236          0.000000   

   sentence_count  exclamation_ratio  warning_word_count  
0             8.0                0.0                 0.0  
1             4.0                0.0                 0.0  
2             5.0                0.0                 0.0  
3             5.0                0.0                 0.0  
4             5.0                0.0                 0.0  


### 6.4 Kiểm tra đặc trưng thủ công theo từng nhãn

In [4]:
handcrafted_with_label = handcrafted.copy()
handcrafted_with_label['label'] = df['label'].values

label_order = ['minimum', 'mild', 'moderate', 'severe']
summary = handcrafted_with_label.groupby('label')[
    ['first_person_ratio', 'negation_ratio', 'absolutist_ratio', 'warning_word_count']
].mean().reindex(label_order)

print(summary.round(4))

          first_person_ratio  negation_ratio  absolutist_ratio  \
label                                                            
minimum               0.0687          0.0088            0.0048   
mild                  0.0870          0.0112            0.0052   
moderate              0.0936          0.0093            0.0055   
severe                0.0972          0.0105            0.0061   

          warning_word_count  
label                         
minimum               0.0798  
mild                  0.1690  
moderate              0.0967  
severe                0.2989  


### 6.5 Ghép TF-IDF (giảm chiều) + Đặc trưng thủ công cho XGBoost

In [5]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

# Giảm chiều TF-IDF từ 3000 xuống 300 để tránh overfit khi ghép với ít mẫu
svd = TruncatedSVD(n_components=300, random_state=42)
X_tfidf_reduced = svd.fit_transform(X_tfidf)

print("Kích thước TF-IDF sau giảm chiều:", X_tfidf_reduced.shape)
print("Phương sai giữ lại được:", svd.explained_variance_ratio_.sum().round(3))

# Ghép với đặc trưng thủ công
X_xgboost = np.hstack([X_tfidf_reduced, handcrafted.values])

print("\nKích thước đặc trưng cuối cùng cho XGBoost:", X_xgboost.shape)

Kích thước TF-IDF sau giảm chiều: (3519, 300)
Phương sai giữ lại được: 0.446

Kích thước đặc trưng cuối cùng cho XGBoost: (3519, 307)


## Bổ sung: Feature Selection và Feature Importance

Dự án sử dụng feature selection gián tiếp qua `min_df`, `max_df`, `max_features` của TF-IDF và giảm chiều bằng TruncatedSVD. Phần dưới tạo báo cáo hệ số Logistic Regression, importance của XGBoost và permutation importance cho 7 handcrafted features.

In [ ]:
RUN_FEATURE_IMPORTANCE = False  # Đổi thành True để chạy lại phân tích

if RUN_FEATURE_IMPORTANCE:
    import subprocess
    import sys
    from pathlib import Path

    PROJECT_ROOT = Path('..').resolve()
    subprocess.run(
        [sys.executable, 'src/feature_selection_importance.py'],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_FEATURE_IMPORTANCE=True để chạy và tạo các báo cáo feature importance.')


In [ ]:
from pathlib import Path
import pandas as pd

report_dir = Path('../outputs/reports')
for filename in [
    'logreg_top_features_by_class.csv',
    'xgboost_importance_aggregated.csv',
    'handcrafted_permutation_importance.csv',
]:
    path = report_dir / filename
    if path.exists():
        print(f'\n{filename}')
        display(pd.read_csv(path).head(20))
